In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 17, The tilde calculus

Companion notebook to [17_tilde_calculus.md](17_tilde_calculus.md). The tilde calculus dualises `ι_X`, `L_X`, `d` on a Poisson manifold `(M, π)` to operators `ι̃_ω`, `L̃_ω`, `d̃` acting on *multivector* fields, indexed by *forms* (and `π`). Six Cartan-style identities mirror the form-side ones; this notebook walks the operator atoms, the three defining rewrites, and `prove_tilde_cartan_relation`.

## The three operator atoms

`tilde_interior(ω)` lowers multivector degree by 1, `tilde_d(π)` raises by 1, `tilde_lie(ω, π)` preserves it.

In [ ]:
from jacopy.calculus.tilde import tilde_interior, tilde_d, tilde_lie
from jacopy.core.expr import Symbol

omega = Symbol("ω")
pi    = Symbol("π")

i_til = tilde_interior(omega)
d_til = tilde_d(pi)
L_til = tilde_lie(omega, pi)

print(f"{i_til}: degree {i_til._degree}")
print(f"{d_til}: degree {d_til._degree}")
print(f"{L_til}: degree {L_til._degree}")

## Defining-identity rewrites

Three rules in `jacopy.calculus.tilde.axioms` realise the defining identities:

* `TildeIotaSwapDefinition`, `ι̃_ω V → ι_V ω` (notation swap)
* `TildeExteriorDLichnerowiczDefinition(π)`, `d̃ V → [π, V]_SN`
* `TildeLieMagicDefinition(π)`, `L̃_ω V → d̃ ι̃_ω V + ι̃_ω d̃ V`

In [ ]:
from jacopy.algebra.derivation import Act
from jacopy.calculus.tilde import (
    TildeIotaSwapDefinition,
    TildeExteriorDLichnerowiczDefinition,
    TildeLieMagicDefinition,
)
from jacopy.proof.expansion import ExpansionEngine

V = Symbol("V")

engine = ExpansionEngine([
    TildeIotaSwapDefinition(),
    TildeExteriorDLichnerowiczDefinition(pi),
    TildeLieMagicDefinition(pi),
])
for inp in [Act(i_til, V), Act(d_til, V), Act(L_til, V)]:
    out, steps = engine.expand(inp)
    print(f"{inp} → {out}  via {steps[0].rule}")

## Auxiliary axioms, `Poisson` flag unlocks `d̃² = 0`

Five auxiliary rules in `tilde.aux_axioms` cover the special cases the three defining rules don't reach:

| Rule | Folds |
|---|---|
| `TildeIotaOnZeroVectorDefinition` | `ι̃_ω f → 0` (f deg 0) |
| `TildeIotaSquaredZeroDefinition` | `ι̃_ω(ι̃_ω V) → 0` |
| `TildeLieOnZeroVectorDefinition` | `L̃_ω f → π^♯(ω)(f)` |
| `TildeDOfFunctionDefinition` | `d̃ f → −π^♯(df)` |
| `TildeDSquaredPoissonDefinition` | `d̃² V → 0` (π Poisson) |

In [ ]:
from jacopy.calculus.tilde import TildeDSquaredPoissonDefinition
from jacopy.core.properties import Poisson
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
reg.declare(pi, Poisson())

engine = ExpansionEngine([
    TildeDSquaredPoissonDefinition(pi, registry=reg),
])
expr = Act(d_til, Act(d_til, V))
out, steps = engine.expand(expr)
print(f"{expr} → {out}")
print(f"rule  : {steps[0].rule}")

## `prove_tilde_cartan_relation`, Cartan magic on a 1-vector

`tilde_intrinsic_engine(pi, koszul, …)` bundles every rule above plus standard MultiEval / Sharp / Pairing helpers. Pair it with `prove_tilde_cartan_relation` and the magic formula closes mechanically. The `slot_kind="covector"` discipline keeps the engine routed to *tilde* rules.

In [ ]:
from jacopy.brackets.koszul import KoszulBracket
from jacopy.calculus.musical import Sharp
from jacopy.calculus.tilde import (
    tilde_intrinsic_engine, prove_tilde_cartan_relation,
)
from jacopy.core.expr import Sum
from jacopy.core.properties import Graded

reg = PropertyRegistry()
reg.declare(pi,    Graded(degree=1)); reg.declare(pi, Poisson())
reg.declare(omega, Graded(degree=1))
reg.declare(V,     Graded(degree=1))
eta = Symbol("η"); reg.declare(eta, Graded(degree=1))

sharp  = Sharp(pi)
koszul = KoszulBracket(sharp)
eng    = tilde_intrinsic_engine(pi, koszul, sharp=sharp, registry=reg)

lhs = Act(L_til, V)
rhs = Sum(Act(d_til, Act(i_til, V)), Act(i_til, Act(d_til, V)))

chain = prove_tilde_cartan_relation(
    lhs, rhs, etas=(eta,), engine=eng, registry=reg,
)
print(f"L̃_ω = d̃ ι̃_ω + ι̃_ω d̃ closes in {len(chain)} steps")

The anti-commute relation `ι̃_ω(ι̃_μ V) + ι̃_μ(ι̃_ω V) = 0` (relation 1 in §3.1.3) closes through the same machinery.

In [ ]:
from jacopy.core.expr import Integer

mu = Symbol("μ");  reg.declare(mu, Graded(degree=1))
W  = Symbol("W");  reg.declare(W,  Graded(degree=2))
i_mu = tilde_interior(mu)

lhs = Sum(Act(i_til, Act(i_mu, W)), Act(i_mu, Act(i_til, W)))
chain = prove_tilde_cartan_relation(
    lhs, Integer(0), etas=(eta,), engine=eng, registry=reg,
)
print(f"ι̃ anti-commute closes in {len(chain)} steps")

## `K̃_η`, the tilde Cartan remainder

`K̃_η := −L̃_η + d̃ ∘ ι̃_η` is the polarity-flipped magic formula. The atom is inert; `TildeCartanRemainderDefinition` realises the defining expansion. Two `K̃` operators on different bivectors stay distinct, useful for deformation arguments.

In [ ]:
from jacopy.calculus.cartan_remainder_axioms import (
    TildeCartanRemainderDefinition,
)
from jacopy.calculus.tilde import K_tilde

K = K_tilde(eta, pi)
print(f"{K}: degree {K._degree}, form {K.form}, bivector {K.bivector}")

engine = ExpansionEngine([TildeCartanRemainderDefinition()])
expr = Act(K, V)
out, steps = engine.expand(expr)
print(f"{expr} → {out}")
print(f"rule  : {steps[0].rule}")

## Summary

* `tilde_interior(ω)`, `tilde_d(π)`, `tilde_lie(ω, π)`, opaque `Derivation` atoms acting on multivectors.
* Three defining rewrites (`TildeIotaSwap`, `TildeExteriorDLichnerowicz`, `TildeLieMagic`) plus five auxiliary rules close the engine machinery.
* `tilde_intrinsic_engine` + `prove_tilde_cartan_relation` close magic in 7 steps and ι̃ anti-commute in 12. `slot_kind="covector"` keeps the tilde and form-side engines from aliasing.
* The `Poisson` flag unlocks `d̃² V → 0` in 4 engine steps, same flag drives the SN-bracket Jacobi chain (tutorial 12).
* `K̃_η` and `TildeCartanRemainderDefinition` are the polarity-flipped shortcut for §3.1.4 derived identities; the atom keys on `(form, bivector)` so multiple `K̃` operators coexist without aliasing.